## Refinamiento de hiperparametros con Optuna

### Imports

In [1]:
import optuna
from ultralytics import YOLO
import torch
from pathlib import Path
import sys

root_path = Path.cwd().parent
if str(root_path) not in sys.path:
    sys.path.append(str(root_path))
    
from config import YAML_PATH, CUSTOM_MODEL_WEIGHTS_PATH

c:\ProgramacionVSC\TFG_ALPR\Sistema-de-Reconocimiento-Automatico-de-Matriculas\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ITERACIONES_OPTUNA = 50

storage_url = "sqlite:///alpr_optimization.db"

### Funcion objetivo

In [3]:
_current_trial = None
# --- CALLBACK PARA CONECTAR YOLO CON OPTUNA ---
def optuna_callback(trainer):
    # Accedemos al trial que guardaremos dentro del objeto trainer
    if hasattr(trainer, 'optuna_trial'):
        global _current_trial
        if _current_trial is not None:
            
            metrics = trainer.validator.metrics.results_dict
            current_score = metrics.get('metrics/mAP50-95(B)', 0)
            
            epoch = trainer.epoch
            _current_trial.report(current_score, step=epoch)
            
            # El Pruner detendrá el trial si los resultados son mediocres
            if _current_trial.should_prune():
                raise optuna.exceptions.TrialPruned()

def objective(trial):
    global _current_trial
    _current_trial = trial
    #Hiperparámetros
    imgsz = trial.suggest_categorical("imgsz", [640, 960, 1280])
    
    #Ajustamos el batch según la resolución para evitar OOM (Out of Memory)
    if imgsz == 1280:
        batch = trial.suggest_categorical("batch_1280", [4, 8])
    elif imgsz == 960:
        batch = trial.suggest_categorical("batch_960", [8, 16])
    else:
        batch = trial.suggest_categorical("batch_640", [16, 32])
        
    lr0 = trial.suggest_float("lr0", 1e-4, 5e-3, log=True)
    lrf = trial.suggest_float("lrf", 0.01, 0.1)
    momentum = trial.suggest_float("momentum", 0.7, 0.98)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)
    
    #Hiperparámetros de aumento 
    mosaic = trial.suggest_float("mosaic", 0.3, 1.0)
    degrees = trial.suggest_float("degrees", 0.0, 10.0)
    scale = trial.suggest_float("scale", 0.3, 0.6)
    
    model = YOLO(CUSTOM_MODEL_WEIGHTS_PATH)
    
    # 3. Añadir el callback manualmente al modelo
    model.add_callback("on_fit_epoch_end", optuna_callback)

    #Entrenar el modelo con los hiperparámetros sugeridos
    try:
        results = model.train(
            data=YAML_PATH,
            epochs=40,
            imgsz=imgsz,
            lr0=lr0,
            lrf=lrf,
            momentum=momentum,
            weight_decay=weight_decay,
            batch=batch,
            mosaic=mosaic,
            workers=2,
            degrees=degrees,
            scale=scale,
            device='cuda' if torch.cuda.is_available() else 'cpu',
            seed=42,
            project="optuna_alpr",
            name=f"trial_{trial.number}",
            exist_ok=True,
            cache=True,
            plots=False,     
            save=True        
        )
        
        # model.trainer.optuna_trial = trial

        final_map = results.results_dict.get('metrics/mAP50-95(B)', 0)
        
        return final_map
    
    except optuna.exceptions.TrialPruned:
        # Si se poda, Optuna necesita saberlo
        raise optuna.exceptions.TrialPruned()
    except Exception as e:
        print(f"Error en trial {trial.number}: {e}")
        return 0.0
    finally:
        torch.cuda.empty_cache()
    

### Estudio

In [ ]:
sampler = optuna.samplers.TPESampler(
    n_startup_trials=10,
    multivariate=True,
    seed=42
)

study = optuna.create_study(
    study_name="alpr_yolo_optimization",
    direction="maximize",
    storage=storage_url,
    sampler=sampler,
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=10),
    load_if_exists=True
    )
study.optimize(objective, n_trials=ITERACIONES_OPTUNA, show_progress_bar=True)

print(f"Mejor trial: {study.best_trial.number}")
print(f"Mejores parámetros: {study.best_params}")